# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [3]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Godność osoby ludzkiej
2. Pan Wołodyjowski
3. Nie budźcie mnie
4. Szymborska. Kropki, przecinki, papierosy
5. Wspólnie. 20 lat kolektywu Sputnik Photos
6. Rodzeństwo (Ritter, Dene, Voss)
7. Męczeństwo i śmierć Marata
8. Gotyk w Karpatach
9. Fisz Emade Tworzywo w Studio
10. Nie ma jak w domu. Projekt fotograficzny Agaty Mendziuk

Czas wykonania: 2.66s


In [4]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Gwałtu, co się dzieje
2. Wspólnie. 20 lat kolektywu Sputnik Photos
3. Z literaturą przez wieki
4. Nie budźcie mnie
5. Dyskusyjny Klub Po Pracy. Poezja smoleńska
6. Nie ma jak w domu. Projekt fotograficzny Agaty Mendziuk
7. Rodzeństwo (Ritter, Dene, Voss)
8. Męczeństwo i śmierć Marata
9. Pan Wołodyjowski
10. Fisz Emade Tworzywo w Studio

Czas wykonania (wątki): 0.63s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [5]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [6]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 10 procesach (rdzeniach)...
Zakończono w czasie 0.37s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [ ]:
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"
NUM_FACTS = 20

def fetch_cat_fact(_):
    return requests.get(CAT_API_URL).json().get('fact')

print("=== Sekwencyjnie ===")
start = time.time()
facts_seq = [fetch_cat_fact(i) for i in range(NUM_FACTS)]
t_seq = time.time() - start
print(f"Pobrano {len(facts_seq)} faktów w {t_seq:.2f}s")
print("Przykładowe fakty:")
for i, fact in enumerate(facts_seq[:3], 1):
    print(f"  {i}. {fact}")

print("\n=== Wielowątkowo (ThreadPoolExecutor) ===")
start = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_FACTS) as executor:
    facts_thr = list(executor.map(fetch_cat_fact, range(NUM_FACTS)))
t_thr = time.time() - start
print(f"Pobrano {len(facts_thr)} faktów w {t_thr:.2f}s")
print("Przykładowe fakty:")
for i, fact in enumerate(facts_thr[:3], 1):
    print(f"  {i}. {fact}")

print(f"\nPrzyspieszenie: {t_seq / t_thr:.1f}x")


=== Sekwencyjnie ===
Pobrano 40 faktów w 13.77s
Przykładowe fakty:
  1. A group of cats is called a “clowder.”
  2. The longest living cat on record according to the Guinness Book belongs to the late Creme Puff of Austin, Texas who lived to the ripe old age of 38 years and 3 days!
  3. Milk can give some cats diarrhea.

=== Wielowątkowo (ThreadPoolExecutor) ===
Pobrano 40 faktów w 1.26s
Przykładowe fakty:
  1. Female felines are \superfecund
  2. Cats lap liquid from the underside of their tongue, not from the top.
  3. A cat has more bones than a human; humans have 206, but the cat has 230 (some cites list 245 bones, and state that bones may fuse together as the cat ages).

Przyspieszenie: 10.9x


### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [9]:
import queue
import threading
import time

NUM_ITEMS = 20

main_q = queue.Queue()
even_q = queue.Queue()
odd_q = queue.Queue()

results_even = []
results_odd = []

def producer():
    for i in range(1, NUM_ITEMS + 1):
        main_q.put(i)
        print(f"[Producent]          Dodano do kolejki: {i}")
        time.sleep(0.05)
    main_q.put(None)  

def dispatcher():
    """Odbiera liczby z głównej kolejki i kieruje do właściwej kolejki konsumenta."""
    while True:
        item = main_q.get()
        if item is None:
            even_q.put(None)
            odd_q.put(None)
            break
        if item % 2 == 0:
            even_q.put(item)
        else:
            odd_q.put(item)

def consumer_even():
    while True:
        item = even_q.get()
        if item is None:
            break
        results_even.append(item)
        print(f"  [Konsument PARZYSTE]   Przetworzono: {item}")

def consumer_odd():
    while True:
        item = odd_q.get()
        if item is None:
            break
        results_odd.append(item)
        print(f"  [Konsument NIEPARZYSTE] Przetworzono: {item}")

threads = [
    threading.Thread(target=producer),
    threading.Thread(target=dispatcher),
    threading.Thread(target=consumer_even),
    threading.Thread(target=consumer_odd),
]

start = time.time()
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"\nParzyste  ({len(results_even)}): {sorted(results_even)}")
print(f"Nieparzyste ({len(results_odd)}): {sorted(results_odd)}")
print(f"Czas: {time.time() - start:.2f}s")


[Producent]          Dodano do kolejki: 1
  [Konsument NIEPARZYSTE] Przetworzono: 1
[Producent]          Dodano do kolejki: 2
  [Konsument PARZYSTE]   Przetworzono: 2
[Producent]          Dodano do kolejki: 3
  [Konsument NIEPARZYSTE] Przetworzono: 3
[Producent]          Dodano do kolejki: 4
  [Konsument PARZYSTE]   Przetworzono: 4
[Producent]          Dodano do kolejki: 5  [Konsument NIEPARZYSTE] Przetworzono: 5

[Producent]          Dodano do kolejki: 6
  [Konsument PARZYSTE]   Przetworzono: 6
[Producent]          Dodano do kolejki: 7
  [Konsument NIEPARZYSTE] Przetworzono: 7
[Producent]          Dodano do kolejki: 8
  [Konsument PARZYSTE]   Przetworzono: 8
[Producent]          Dodano do kolejki: 9  [Konsument NIEPARZYSTE] Przetworzono: 9

[Producent]          Dodano do kolejki: 10
  [Konsument PARZYSTE]   Przetworzono: 10
[Producent]          Dodano do kolejki: 11
  [Konsument NIEPARZYSTE] Przetworzono: 11
[Producent]          Dodano do kolejki: 12
  [Konsument PARZYSTE]   Przetworz

### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [10]:
import multiprocessing
import time
from lab2_functions import calculate_power_sum

if __name__ == "__main__":
    RANGE_START = 1
    RANGE_END = 10_001 
    cores = multiprocessing.cpu_count()
    print(f"Obliczanie sum potęg dla liczb {RANGE_START}–{RANGE_END - 1} na {cores} procesach...")

    numbers = list(range(RANGE_START, RANGE_END))

    start = time.time()
    seq_results = [calculate_power_sum(n) for n in numbers]
    t_seq = time.time() - start
    print(f"Sekwencyjnie: {t_seq:.2f}s  |  Suma wyników: {sum(seq_results)}")

    start = time.time()
    with multiprocessing.Pool(processes=cores) as pool:
        mp_results = pool.map(calculate_power_sum, numbers)
    t_mp = time.time() - start
    print(f"Wieloprocesowo: {t_mp:.2f}s  |  Suma wyników: {sum(mp_results)}")

    print(f"\nPrzyspieszenie: {t_seq / t_mp:.1f}x")
    print(f"Wyniki zgodne: {seq_results == mp_results}")


Obliczanie sum potęg dla liczb 1–10000 na 10 procesach...
Sekwencyjnie: 0.26s  |  Suma wyników: 995207854197964442547664779149713094406225681179265198421228880588357806296714186703210526583761834998114384212288730396901480433958551919996123240783736004317692688469399493136990101249303140987388533721044806766065437006409106438027712620589642097820975739314540123609570363583485256201869594157997582865282767878851852797262927388912728290810611143129578917963952248558896652100510662230361518505000
Wieloprocesowo: 0.12s  |  Suma wyników: 995207854197964442547664779149713094406225681179265198421228880588357806296714186703210526583761834998114384212288730396901480433958551919996123240783736004317692688469399493136990101249303140987388533721044806766065437006409106438027712620589642097820975739314540123609570363583485256201869594157997582865282767878851852797262927388912728290810611143129578917963952248558896652100510662230361518505000

Przyspieszenie: 2.2x
Wyniki zgodne: True
